# EfficientNetB0

Candidato a estudiante. 4.88M parámetros, 104 ms/imagen.

In [ ]:
# En Kaggle clona el repo; en local no hace nada.
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/CCSandoval/ButterflyModeling.git"

if Path("/kaggle").exists():
    CLON = "/tmp/ButterflyModeling"
    if not Path(CLON).exists():                     # el repo es público: clonar no pide credenciales
        subprocess.run(["git", "clone", "--depth", "1", REPO, CLON], check=True)
    sys.path.insert(0, CLON)
    os.chdir(CLON)
    print("clon en", CLON, "·", subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], cwd=CLON,
        capture_output=True, text=True).stdout.strip())

    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        print("sin GITHUB_TOKEN: no se publicará")

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

In [ ]:
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import tensorflow as tf

from scripts import corpus, evalStats
from scripts import modelRegistry as registro
from scripts.architectures import IMG_SIZE, buildModel, descongelarCola, preprocessFn
from scripts.entrenamiento import callbacksEntrenamiento
from scripts.dataPipeline import aumentadorFotometrico, loadSplit, preparar
from scripts.vizStyle import SERIE_1, SERIE_2, applyStyle, plotConfusionMatrix

ARQUITECTURA = "efficientnet_b0"
RUN_NAME = "efficientnet-b0"
NOTAS = "Candidato a estudiante: efficientnet-b0 + cabeza comun"
ROL = "estudiante"

# 0.0 reproduce el run con backbone congelado; 0.30 descongela la cola
FRACCION_DESCONGELADA = 0.30

TAMANO = IMG_SIZE
LOTE = 32
EPOCAS = 100
EPOCAS_CALENTAMIENTO = 5
PACIENCIA = 8
TASA_APRENDIZAJE = 1e-4 if FRACCION_DESCONGELADA else 1e-3
SEMILLA = 42

tf.keras.utils.set_random_seed(SEMILLA)
corpus.materializar()

RUN_ID = registro.buildRunId(RUN_NAME)
RUN_DIR = registro.createRunDir(RUN_ID)
IMGS = RUN_DIR / "imgs"

print(f"run_id : {RUN_ID}")
print(f"TF     : {tf.__version__} · dispositivos: "
      f"{[d.device_type for d in tf.config.list_physical_devices()]}")

## 1 · Datos

Un tf.data.Dataset por split. Solo `train` se aumenta; `validate`/`test` son
deterministas (`shuffle=False`, indispensable para que `y_true` se alinee con
las predicciones).

In [ ]:
preprocess = preprocessFn(ARQUITECTURA)
augmentador = aumentadorFotometrico()

dsTrainCrudo, clases = loadSplit("dataset/train", TAMANO, LOTE)
dsValCrudo, _ = loadSplit("dataset/validate", TAMANO, LOTE, shuffle=False)
dsTestCrudo, _ = loadSplit("dataset/test", TAMANO, LOTE, shuffle=False)

dsTrain = preparar(dsTrainCrudo, preprocess, augmentador)
dsVal = preparar(dsValCrudo, preprocess)
dsTest = preparar(dsTestCrudo, preprocess)

print(f"{len(clases)} clases")

## 2 · Modelo

In [ ]:
modelo = buildModel(ARQUITECTURA, len(clases))

entrenables = sum(int(np.prod(w.shape)) for w in modelo.trainable_weights)
congelados = sum(int(np.prod(w.shape)) for w in modelo.non_trainable_weights)
pd.Series({
    "parametros_totales": modelo.count_params(),
    "entrenables": entrenables,
    "congelados": congelados,
    "clases_salida": len(clases),
}).to_frame("valor")

## 3 · Pesos de clase

In [ ]:
pesosClase = corpus.classWeights()
pd.Series(
    {clases[i]: round(p, 3) for i, p in pesosClase.items()}
).sort_values(ascending=False).head(8).to_frame("peso")

## 4 · Entrenamiento

In [ ]:
from tensorflow.keras.optimizers import Adam

modelo.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=TASA_APRENDIZAJE),
    metrics=["accuracy"],
)

inicio = datetime.now(timezone.utc)

calentamiento = {}
if FRACCION_DESCONGELADA:
    calentamiento = modelo.fit(
        dsTrain,
        validation_data=dsVal,
        epochs=EPOCAS_CALENTAMIENTO,
        class_weight=pesosClase,
        verbose=2,
    ).history
    print(descongelarCola(modelo, FRACCION_DESCONGELADA))
    modelo.compile(
        loss="categorical_crossentropy",
        optimizer=Adam(learning_rate=TASA_APRENDIZAJE),
        metrics=["accuracy"],
    )

# reanudables: un corte cuesta una época, no el run entero
callbacks = callbacksEntrenamiento(RUN_NAME, PACIENCIA)

ajuste = modelo.fit(
    dsTrain,
    validation_data=dsVal,
    epochs=EPOCAS,
    callbacks=callbacks,
    class_weight=pesosClase,
    verbose=2,
)
fin = datetime.now(timezone.utc)

epocasCalentadas = len(calentamiento.get("loss", []))
historia = {}
for clave, valores in ajuste.history.items():
    previo = calentamiento.get(clave)
    historia[clave] = ([float(v) for v in previo] if previo else [None] * epocasCalentadas)
    historia[clave] += [float(v) for v in valores]
print(f"\n{len(historia['loss'])} épocas en {(fin - inicio).total_seconds() / 60:.1f} min")

## 5 · Curvas de entrenamiento

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

epocas = range(1, len(historia["loss"]) + 1)
figura, ejes = plt.subplots(1, 2, figsize=(11, 4.2))

ejes[0].plot(epocas, historia["accuracy"], color=SERIE_1, label="Entrenamiento")
ejes[0].plot(epocas, historia["val_accuracy"], color=SERIE_2, label="Validación")
ejes[0].set(title=f"{RUN_NAME} · Exactitud", xlabel="Época", ylim=(0, 1))
ejes[0].legend(loc="lower right")

ejes[1].plot(epocas, historia["loss"], color=SERIE_1, label="Entrenamiento")
ejes[1].plot(epocas, historia["val_loss"], color=SERIE_2, label="Validación")
ejes[1].set(title=f"{RUN_NAME} · Pérdida", xlabel="Época")
ejes[1].legend(loc="upper right")

figura.tight_layout()
figura.savefig(IMGS / "training_history.png", bbox_inches="tight")
plt.show()

## 6 · Evaluación

Dos conjuntos: `test` (20%, métrica principal) y `validate` (10%, el que
monitorean los callbacks).

In [ ]:
metricas = {
    "test": evalStats.evaluar(modelo, dsTest, clases),
    "validate": evalStats.evaluar(modelo, dsVal, clases),
}

pd.DataFrame([
    {k: round(v, 4) if isinstance(v, float) else v
     for k, v in m.items()
     if k in ("num_samples", "accuracy", "top3_accuracy",
              "macro_f1", "weighted_f1", "macro_precision", "macro_recall")}
    for m in metricas.values()
], index=list(metricas.keys()))

## 7 · Matriz de confusión

In [ ]:
_ = plotConfusionMatrix(
    metricas["test"],
    f"Matriz de confusión · prueba · {RUN_NAME}",
)
plt.savefig(IMGS / "confusion_test.png", bbox_inches="tight")
plt.show()

## 8 · Confusiones principales

In [ ]:
confusiones = evalStats.confusionesPrincipales(metricas["test"])
pd.DataFrame(confusiones).head(10)

## 9 · Versionado

In [ ]:
modelo.save(RUN_DIR / "model.keras")
registro.writeJson(RUN_DIR / "history.json", historia)
registro.writeJson(RUN_DIR / "metrics.json", {
    "run_id": RUN_ID,
    "class_names": clases,
    "splits": metricas,
    "top_confusions": {
        s: evalStats.confusionesPrincipales(m) for s, m in metricas.items()
    },
})

configuracion = {
    "run_id": RUN_ID,
    "name": RUN_NAME,
    "dataset": corpus.manifiesto()["creado"],
    "architecture": ARQUITECTURA,
    "role": ROL,
    "notes": NOTAS,
    "num_classes": len(clases),
    "hyperparameters": {
        "img_size": list(TAMANO),
        "batch_size": LOTE,
        "epochs_budget": EPOCAS,
        "epochs_warmup": EPOCAS_CALENTAMIENTO if FRACCION_DESCONGELADA else 0,
        "epochs_ran": len(historia["loss"]),
        "learning_rate": TASA_APRENDIZAJE,
        "early_stopping_patience": PACIENCIA,
        "class_weights": "balanced",
        "seed": SEMILLA,
        "finetuning": FRACCION_DESCONGELADA,
        "backbone": f"{ARQUITECTURA}/imagenet",
    },
    "environment": {
        "commit": registro.commitActual(),
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "platform": platform.platform(),
        "acelerador": next((d.device_type for d in tf.config.list_physical_devices()
                            if d.device_type != "CPU"), "cpu"),
    },
    "started_at": inicio.isoformat(),
    "finished_at": fin.isoformat(),
    "training_minutes": round((fin - inicio).total_seconds() / 60, 1),
}
registro.writeJson(RUN_DIR / "run.json", configuracion)

registro.registrar(RUN_ID)

print(f"Run guardado en {RUN_DIR}")

if Path("/kaggle").exists() and os.environ.get("GITHUB_TOKEN"):
    from scripts.kaggle import publicarRun

    publicarRun(RUN_ID)


## Resumen

In [ ]:
pd.Series({
    "run_id": RUN_ID,
    "arquitectura": ARQUITECTURA,
    "clases": len(clases),
    "epocas": len(historia["loss"]),
    "minutos": configuracion["training_minutes"],
    "test_accuracy": round(metricas["test"]["accuracy"], 4),
    "test_top3": round(metricas["test"]["top3_accuracy"], 4),
    "test_macro_f1": round(metricas["test"]["macro_f1"], 4),
    "validate_accuracy": round(metricas["validate"]["accuracy"], 4),
    "validate_macro_f1": round(metricas["validate"]["macro_f1"], 4),
}).to_frame("valor")